# IEEE-CIS Fraud Detection — Final Production Model

## Goal
Every prior notebook used `train_features.parquet` for fitting and `val_features.parquet`
for evaluation, early stopping, hyperparameter tuning, threshold selection, and SHAP analysis.
That validation set has now done its job.

Before deployment, retrain on **train + val combined** so the production model:
- Uses all available labeled data (more signal)
- Is trained up to the most recent transaction date we have (val is the most recent window before the held-out test set) — reduces the train/production time gap

## The early-stopping problem
`lightgbm_final.pkl` was trained with early stopping monitored on val. Val can't be both
training data and the stopping signal. Fix: drop early stopping and train for a **fixed**
number of iterations — the `best_iteration_` (379) found when `lightgbm_final.pkl` was trained.

## Output
`models/lightgbm_production.pkl` — the model Phase 7's FastAPI service will load.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier, log_evaluation

os.chdir('/Users/shaliqshukoor/fraud-risk-system')

# Load train + val
train_df = pd.read_parquet('data/processed/train_features.parquet')
val_df   = pd.read_parquet('data/processed/val_features.parquet')

with open('data/processed/selected_feature_names.json') as f:
    selected_features = json.load(f)

full_df = pd.concat([train_df, val_df], axis=0).sort_values('TransactionDT').reset_index(drop=True)

X_full = full_df[selected_features].copy()
y_full = full_df['isFraud']

# Label encode object columns (fit on the full combined set this time —
# this IS the final training set, there's no more held-out split to leak into)
obj_cols = X_full.select_dtypes(include='object').columns.tolist()
encoders = {}
for col in obj_cols:
    le = LabelEncoder()
    X_full[col] = le.fit_transform(X_full[col].astype(str))
    encoders[col] = le

print(f'Combined train+val: {X_full.shape}')
print(f'Fraud rate: {y_full.mean():.4f}')
print(f'Object columns label-encoded: {obj_cols}')

## Chapter 1 — Retrain on Combined Data, Fixed Iterations

Use the same Optuna-tuned hyperparameters as `lightgbm_final.pkl`, but:
- `n_estimators = 379` (the `best_iteration_` from the earlier early-stopped run)
- No `eval_set`, no early stopping — the iteration count is fixed, so there's nothing to monitor against

In [ ]:
study = joblib.load('models/optuna_study_lgbm.pkl')

final_params = {
    **study.best_params,
    'n_estimators': 379,  # best_iteration_ from lightgbm_final.pkl's early-stopped run
    'is_unbalance': True,
    'metric': 'average_precision',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}
print('Final production params:')
for k, v in final_params.items():
    print(f'  {k}: {v}')

production_model = LGBMClassifier(**final_params)
production_model.fit(X_full, y_full, callbacks=[log_evaluation(100)])

print('\nTraining complete.')
print(f'Trees built: {production_model.n_estimators_}')

## Chapter 2 — Save the Production Model

Save the model, its label encoders, and the feature list together — this is the
exact bundle the FastAPI service (Phase 7) will load to score new transactions.

In [ ]:
joblib.dump(production_model, 'models/lightgbm_production.pkl')
joblib.dump(encoders, 'models/label_encoders.pkl')

with open('data/processed/selected_feature_names.json') as f:
    print(f'Feature list (412 features) reused as-is: data/processed/selected_feature_names.json')

print('Saved models/lightgbm_production.pkl')
print('Saved models/label_encoders.pkl')

## Conclusion

`models/lightgbm_production.pkl` is now the deployment artifact:
- Trained on all 590,540 labeled rows (train + val), sorted by `TransactionDT`
- Same hyperparameters and 412 features validated throughout Phases 3-6
- Fixed 379 trees (no early stopping needed/possible on the full dataset)
- Label encoders saved alongside, so the API can encode raw categorical inputs identically

The held-out **test set** remains completely unused — it's reserved for Phase 6
(monitoring/drift simulation) as a stand-in for "new production traffic."

**Note:** We can't compute AUC-PR/profit for this model in the usual way — it was trained
on the only labeled data we'd normally evaluate against. That's expected and correct:
all of `lightgbm_final.pkl`'s validated metrics (AUC-PR 0.5435, profit $484,385 @ threshold 0.20)
transfer as our expectation for this model's performance, since it's the same architecture
and hyperparameters trained on a superset of the same data.